In [1]:
import sys
import os

# Aggiungi project root al path per gli import
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project Root aggiunta al path: {project_root}")

Project Root aggiunta al path: c:\Users\emagi\Documents\Deep_Learning\Progetto_deep_learning


In [2]:
# CALCOLO NAIVE MAE PER I 3 FOLD DI VALIDAZIONE 
# Valido sia per Sliding Window che per Expanding Window (hanno gli stessi
# mesi di validazione, stessi campioni senza shuffle)
import pandas as pd
import torch
import torch.nn as nn
from src.ModelClasses.naive import NaivePersistence
from src.Training.engine import validate_one_epoch
from src.DataLoading.data_loader import TS_Cross_Validator
from src.config import TARGET_COL, SAMPLING_CONFIG, NAIVE_CONFIG

# 1. Caricamento Dati 
df = pd.read_csv("../data/processed/preprocessed_ds.csv")
print(f"Dataset shape: {df.shape}")
print(f"Target column: {TARGET_COL}")

# 2. Creazione Folds
validator = TS_Cross_Validator(df, TARGET_COL, SAMPLING_CONFIG)
folds = list(validator.get_folds())  # Converti generator in lista

# 3. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Usiamo L1Loss (MAE) per il denominatore del MASE
mae_metric = nn.L1Loss()

naive_maes = []

print(f"\n--- Calcolo Benchmark Naive su {len(folds)} Fold ---\n")

# Crea modello Naive con model_config (non usa più train_loader)
naive_model = NaivePersistence(model_config=NAIVE_CONFIG).to(device)

for i, (train_loader, val_loader, scaler) in enumerate(folds):
    # validate_one_epoch restituisce 3 valori: (avg_loss, avg_mae, avg_rmse)
    _, fold_mae, _ = validate_one_epoch(naive_model, val_loader, nn.MSELoss(), device)
    
    naive_maes.append(fold_mae)
    print(f"FOLD {i+1} -> Naive MAE: {fold_mae:.6f}")

print("\n--- COPIA QUESTO NEL TUO training_config.py ---")
print(f"NAIVE_MAE_PER_FOLD = {naive_maes}")

c:\Users\emagi\Documents\Deep_Learning\Progetto_deep_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset shape: (17544, 24)
Target column: pv_power

=== INIZIO CROSS-VALIDATION (3 splits) ===

---------------- FOLD 1 ----------------
TRAIN: 0 -> 8783
VAL  : 8760 -> 11703
Fold 1 pronto. Yielding...

---------------- FOLD 2 ----------------
TRAIN: 0 -> 11703
VAL  : 11680 -> 14623
Fold 2 pronto. Yielding...

---------------- FOLD 3 ----------------
TRAIN: 0 -> 14623
VAL  : 14600 -> 17543
Fold 3 pronto. Yielding...
Device: cuda

--- Calcolo Benchmark Naive su 3 Fold ---

NaivePersistence - target_idx: 23, horizon: 24
FOLD 1 -> Naive MAE: 0.062281
FOLD 2 -> Naive MAE: 0.086214
FOLD 3 -> Naive MAE: 0.061930

--- COPIA QUESTO NEL TUO training_config.py ---
NAIVE_MAE_PER_FOLD = [0.06228089907571026, 0.08621430409181377, 0.06193031994221003]


In [3]:
# CALCOLO NAIVE MAE PER FINAL FOLD (22 mesi train + 2 mesi val)

from src.DataLoading import create_final_train_val_loaders
# Crea train/val loaders per il final fold
_, final_val_loader, _ = create_final_train_val_loaders(
    df, target_col=TARGET_COL
)
# Calcola MAE del modello naive sul final validation fold
_, final_fold_mae, _ = validate_one_epoch(
    naive_model, final_val_loader, nn.MSELoss(), device
)
print("\n--- FINAL FOLD (22 mesi train + 2 mesi val) ---")
print(f"Naive MAE: {final_fold_mae:.6f}")
print("\n--- COPIA QUESTO IN src/config/training_config.py ---")
print(f"NAIVE_MAE_FINAL_FOLD = {final_fold_mae}")


=== FINAL TRAINING SPLIT ===
Train: 16081 rows -> 16010 samples
Val:   1463 rows -> 1440 samples

--- FINAL FOLD (22 mesi train + 2 mesi val) ---
Naive MAE: 0.049197

--- COPIA QUESTO IN src/config/training_config.py ---
NAIVE_MAE_FINAL_FOLD = 0.04919686427582865


In [ ]:
# CALCOLO NAIVE MAE SUL TEST SET
# Esegui questa cella DOPO aver ricevuto i dati di test

from src.DataLoading import create_test_loader
from src.PreProcessing.preprocessing import Preprocesser, PREPROCESS_CONFIG
from src.config import LOOKBACK, HORIZON

# Configurazione
TEST_DATA_PATH = "../data/processed/merged_test_ds.csv"  # Modifica con il path del test set

# 1. Carica e preprocessa test data
if TEST_DATA_PATH.endswith(('.xlsx', '.xls')):
    test_raw = pd.read_excel(TEST_DATA_PATH)
else:
    test_raw = pd.read_csv(TEST_DATA_PATH)
print(f"Test raw shape: {test_raw.shape}")

preprocesser = Preprocesser(test_raw, PREPROCESS_CONFIG)
test_df = preprocesser.run()
test_df = test_df[df.columns]  # Riordina colonne come training
print(f"Test preprocessed shape: {test_df.shape}")

# 2. Crea test loader (usa df dal training già caricato sopra)
test_loader, _ = create_test_loader(
    train_df=df,
    test_df=test_df,
    target_col=TARGET_COL,
    lookback=LOOKBACK,
    horizon=HORIZON,
)

# 3. Calcola Naive MAE sul test set
_, test_naive_mae, _ = validate_one_epoch(
    naive_model, test_loader, nn.MSELoss(), device
)

print("\n--- TEST SET ---")
print(f"Naive MAE: {test_naive_mae:.6f}")
print("\n--- COPIA QUESTO IN src/config/training_config.py ---")
print(f"NAIVE_MAE_TEST = {test_naive_mae}")

Test raw shape: (17544, 16)
Test preprocessed shape: (17544, 24)

=== TEST DATA LOADER ===
Test: 17544 rows -> 17497 samples

--- TEST SET ---
Naive MAE: 0.067466

--- COPIA QUESTO IN src/config/training_config.py ---
NAIVE_MAE_TEST = 0.06746598157732592
